In [14]:
import pandas as pd

test_df = pd.read_csv('../data/original_data/ais_test.csv')
schedules_df = pd.read_csv('../data/original_data/schedules_to_may_2024.csv', sep='|')

test_df.head()

,ID,vesselId,time,scaling_factor
0,0,61e9f3aeb937134a3c4bfe3d,2024-05-08 00:03:16,0.3
1,1,61e9f473b937134a3c4c02df,2024-05-08 00:06:17,0.3
2,2,61e9f469b937134a3c4c029b,2024-05-08 00:10:02,0.3
3,3,61e9f45bb937134a3c4c0221,2024-05-08 00:10:34,0.3
4,4,61e9f38eb937134a3c4bfd8d,2024-05-08 00:12:27,0.3


In [15]:
schedules_df.head()

,vesselId,shippingLineId,shippingLineName,arrivalDate,sailingDate,portName,portId,portLatitude,portLongitude
0,61e9f3b1b937134a3c4bfe53,61a8e672f9cba188601e84ac,Wallenius Wilhelmsen Ocean,2023-10-02 00:00:00+00:00,2023-10-03 00:00:00+00:00,Port of Brunswick,61d38499b7b7526e1adf3d54,31.140556,-81.496667
1,61e9f3b1b937134a3c4bfe53,61a8e672f9cba188601e84ac,Wallenius Wilhelmsen Ocean,2023-10-27 00:00:00+00:00,2023-10-27 00:00:00+00:00,Port of Southampton,61d3832bb7b7526e1adf3b63,50.902500,-1.428889
2,61e9f3b1b937134a3c4bfe53,61a8e672f9cba188601e84ac,Wallenius Wilhelmsen Ocean,2023-10-19 00:00:00+00:00,2023-10-20 00:00:00+00:00,Port of Bremerhaven,61d375e793c6feb83e5eb3e2,53.563611,8.554722
3,61e9f3b1b937134a3c4bfe53,61a8e672f9cba188601e84ac,Wallenius Wilhelmsen Ocean,2023-10-09 00:00:00+00:00,2023-10-10 00:00:00+00:00,Port of New York,61d38481b7b7526e1adf3d23,40.688333,-74.028611
4,61e9f3b1b937134a3c4bfe53,61a8e672f9cba188601e84ac,Wallenius Wilhelmsen Ocean,2023-09-25 00:00:00+00:00,2023-09-26 00:00:00+00:00,Manzanillo International Terminal,61d37d0199db2ccf7339eee1,9.372370,-79.879790


In [16]:
import pandas as pd
import numpy as np
from datetime import datetime

def engineer_port_locations(schedule_df, test_df):
    """
    Engineer port latitude and longitude features for test data based on vessel schedules.
    Handles timezone-aware datetime comparisons.
    
    Parameters:
    schedule_df (pd.DataFrame): Schedule data with vessel ports and times
    test_df (pd.DataFrame): Test data requiring port location features
    
    Returns:
    pd.DataFrame: Test data with added port_lat and port_long features
    """
    # Convert string data to datetime and ensure timezone consistency
    schedule_df = schedule_df.copy()
    test_df = test_df.copy()
    
    # Convert schedule dates to UTC
    schedule_df['arrivalDate'] = pd.to_datetime(schedule_df['arrivalDate']).dt.tz_convert('UTC')
    schedule_df['sailingDate'] = pd.to_datetime(schedule_df['sailingDate']).dt.tz_convert('UTC')
    
    # Convert test dates to UTC
    test_df['time_utc'] = pd.to_datetime(test_df['time']).dt.tz_localize('UTC')
    
    # Initialize new columns
    test_df['port_lat'] = np.nan
    test_df['port_long'] = np.nan
    
    # Process each row in test_df
    for idx, row in test_df.iterrows():
        vessel_schedule = schedule_df[schedule_df['vesselId'] == row['vesselId']].copy()
        
        if len(vessel_schedule) == 0:
            continue
            
        # Sort vessel schedule by arrival date
        vessel_schedule = vessel_schedule.sort_values('arrivalDate')
        
        # Find the relevant port
        # Case 1: Vessel is at a port (between arrival and sailing)
        at_port = vessel_schedule[
            (vessel_schedule['arrivalDate'] <= row['time_utc']) & 
            (vessel_schedule['sailingDate'] >= row['time_utc'])
        ]
        
        if len(at_port) > 0:
            # Use the current port's coordinates
            test_df.at[idx, 'port_lat'] = at_port.iloc[0]['portLatitude']
            test_df.at[idx, 'port_long'] = at_port.iloc[0]['portLongitude']
            continue
        
        # Case 2: Vessel is between ports
        next_port = vessel_schedule[vessel_schedule['arrivalDate'] > row['time_utc']].iloc[0] if len(vessel_schedule[vessel_schedule['arrivalDate'] > row['time_utc']]) > 0 else None
        prev_port = vessel_schedule[vessel_schedule['sailingDate'] < row['time_utc']].iloc[-1] if len(vessel_schedule[vessel_schedule['sailingDate'] < row['time_utc']]) > 0 else None
        
        if prev_port is not None and next_port is not None:
            # Calculate time ratios
            total_time = (next_port['arrivalDate'] - prev_port['sailingDate']).total_seconds()
            elapsed_time = (row['time_utc'] - prev_port['sailingDate']).total_seconds()
            ratio = elapsed_time / total_time
            
            # Interpolate coordinates
            test_df.at[idx, 'port_lat'] = prev_port['portLatitude'] + (next_port['portLatitude'] - prev_port['portLatitude']) * ratio
            test_df.at[idx, 'port_long'] = prev_port['portLongitude'] + (next_port['portLongitude'] - prev_port['portLongitude']) * ratio
        
        elif prev_port is not None:
            # Use last known port
            test_df.at[idx, 'port_lat'] = prev_port['portLatitude']
            test_df.at[idx, 'port_long'] = prev_port['portLongitude']
        
        elif next_port is not None:
            # Use next port
            test_df.at[idx, 'port_lat'] = next_port['portLatitude']
            test_df.at[idx, 'port_long'] = next_port['portLongitude']

    test_df.drop(["time_utc"], axis=1, inplace=True)
    
    return test_df

test_df = engineer_port_locations(test_df=test_df, schedule_df=schedules_df)



In [17]:
test_df.head(50)

,ID,vesselId,time,scaling_factor,port_lat,port_long
0,0,61e9f3aeb937134a3c4bfe3d,2024-05-08 00:03:16,0.3,53.534403,8.367380
1,1,61e9f473b937134a3c4c02df,2024-05-08 00:06:17,0.3,NaN,NaN
2,2,61e9f469b937134a3c4c029b,2024-05-08 00:10:02,0.3,NaN,NaN
3,3,61e9f45bb937134a3c4c0221,2024-05-08 00:10:34,0.3,NaN,NaN
4,4,61e9f38eb937134a3c4bfd8d,2024-05-08 00:12:27,0.3,NaN,NaN
5,5,clh6aqawa0006gh0zje911dl3,2024-05-08 00:12:31,0.3,NaN,NaN
6,6,61e9f3c7b937134a3c4bfedf,2024-05-08 00:13:58,0.3,NaN,NaN
7,7,61e9f3b0b937134a3c4bfe4f,2024-05-08 00:14:41,0.3,51.336389,3.207222
8,8,61e9f446b937134a3c4c01ab,2024-05-08 00:15:03,0.3,NaN,NaN
9,9,61e9f3cbb937134a3c4bff03,2024-05-08 00:15:34,0.3,NaN,NaN


In [18]:
test_df.to_csv("../data/processed_data/test.csv", index=False)